# 68장 · FastAPI 요청 모델과 POST 엔드포인트

FastAPI는 HTTP 요청을 경로 함수에 전달합니다. 요청 모델에서 거부된 입력과 서비스까지 도착해 근거를 찾지 못한 입력의 결과를 비교합니다.

과정 안내 함수를 FastAPI의 POST /ask 경로로 제공합니다. TestClient로 서버 포트를 열지 않고도 요청 검증, HTTP 상태, 답변 근거를 검사합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## Pydantic 요청 모델과 입력 검증

Pydantic 모델은 question과 k를 파싱합니다. 질문 앞뒤 공백은 정리하고 공백만 남은 질문은 거절합니다. k에는 1~10 범위의 정수만 받습니다. 아래의 수업용 서비스는 고정 문서 한 건에서 A1 결과를 만듭니다.


### 셀 01 · FastAPI 앱과 요청 모델

이 셀은 앱과 AskRequest 클래스를 만듭니다. 다음 셀의 @app.post가 경로와 입력 모델을 연결합니다.


In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field, StrictInt, field_validator
app = FastAPI()
class AskRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    question: str = Field(min_length=1, max_length=1000)
    k: StrictInt = Field(default=2, ge=1, le=10)
    @field_validator("question")
    @classmethod
    def strip_question(cls, value):
        if not value.strip():
            raise ValueError("blank_question")
        return value.strip()


AskRequest가 정의됐습니다. 아직 질문은 전송하지 않았습니다.


### POST 요청을 읽는 순서

<p>TestClient는 POST /ask에 JSON 본문을 보냅니다. FastAPI는 먼저 AskRequest로 본문을 검사하고 통과한 모델을 경로 함수 ask에 넘깁니다.</p>


### 공백 질문과 k 검증

<p>question에 공백만 있으면 strip() 결과가 비어서 검증이 실패합니다. k를 생략하면 2를 쓰고, 보내는 경우에는 1~10 사이의 정수여야 합니다. 문자열 "2"는 StrictInt 검사를 통과하지 않습니다.</p>


### 셀 02 · POST 경로와 서비스 함수

question은 JSON 본문으로 보내세요. 함수에 건네는 문자열과 응답의 근거를 확인합니다.

이 짧은 경로에서 question은 검색에 들어가고 k는 요청 형식을 검사하는 데 쓰입니다. k를 바꿨는데 문서 개수가 그대로라면, 검사 뒤 검색 함수까지 그 값이 전달되는지 살펴보세요.


In [2]:
def answer(question):
    source = {"id": "course-guide-v1:p2", "page": 1, "source": "가상 과정 운영 안내",
              "text": "과제는 학습 관리 페이지의 과제함에 제출합니다."}
    if "과제" in question:
        return {"status": "answered", "answer": source["text"] + " [course-guide-v1:p2]",
                "sources": [source], "generation_calls": 1}
    return {"status": "no_evidence", "answer": "자료에서 확인할 수 없습니다.", "sources": [], "generation_calls": 0}

@app.post("/ask")
def ask(body: AskRequest):
    return {"run_status": "completed", "result": answer(body.question)}

client = TestClient(app)
response = client.post("/ask", json={"question": "과제는 어디에 내나요?", "k": 2})
print(response.status_code, response.json()["result"]["sources"][0]["id"])

200 course-guide-v1:p2


경로 함수는 입력 모델이 검사를 마친 뒤 answer(body.question)을 호출합니다. k는 검사하지만 이 한 문서짜리 검색 함수에는 전달하지 않습니다.

응답 상태 200과 p2는 서로 다른 검사를 뒷받침합니다. 200은 요청 처리가 끝났다는 신호이고, p2는 답변의 근거를 추적할 식별자입니다. 요청 처리 테스트가 성공했더라도 문서 질문에 맞는 문단인지 확인하는 검사는 별도로 필요합니다.


### HTTP 200 안의 답변 상태

<p>200은 이 요청에 대한 HTTP 처리가 성공했다는 뜻입니다. 실제 근거를 찾았는지는 result.status와 sources를 보고 판별합니다.</p>


## HTTP 검증 오류와 빈 근거 응답

입력을 읽을 수 없는 경우와, 입력은 정상이지만 근거를 찾지 못한 경우에 화면이 해야 할 일이 다릅니다. 전자는 입력을 고치고 후자는 다른 자료나 질문이 필요한지 안내합니다.


### 셀 03 · 공백 입력의 검증 오류

Pydantic의 입력 검사를 통과하지 못한 요청의 상태 코드를 확인하세요.


In [3]:
bad = client.post("/ask", json={"question": "   "})
print(bad.status_code)

422


422입니다. 질문을 검사하는 단계에서 멈춰 서비스 함수에는 도달하지 않았습니다.


### 검증 실패가 끝나는 지점

<p>공백만 보낸 요청의 422는 검색이 실패했다는 표시가 아닙니다. 입력 모델이 요청을 거부해 서비스 함수 answer에 질문이 전달되지 않았습니다.</p>


### 과제 04 · 유효한 요청의 빈 검색 결과

주차비를 묻는 POST를 보내 HTTP 코드와 답변의 상태를 출력하세요.


In [4]:
# client.post에 주차 질문을 넣으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply = client.post(&#x27;/ask&#x27;, json={&#x27;question&#x27;: &#x27;주차는 무료인가요?&#x27;})
print(reply.status_code, reply.json()[&#x27;result&#x27;][&#x27;status&#x27;])</code></pre>
<p>실행 결과</p><pre><code>200 no_evidence</code></pre>
<p>200 no_evidence입니다. 요청은 정상 처리됐고 제공 자료가 주차비를 다루지 않았습니다.</p>
</details>


### 유효한 주차 질문과 빈 검색 결과

<p>주차 질문은 유효한 문자열이므로 경로 함수를 호출합니다. 과제 문서에 근거가 없어도 검색 경로는 정상 종료하며, JSON에는 no_evidence와 빈 sources가 담깁니다.</p>


## 복습과 실습 점검

### 과제 05 · HTTP 422에서 수정할 입력 필드 찾기

질문은 그대로 두고 k를 문자열로 전송하세요. 상태 코드와 오류가 난 필드 이름을 출력하세요. 이 두 값으로 사용자가 어느 입력을 고쳐야 하는지 알 수 있습니다.


In [ ]:
# k를 "2"로 보내고 응답의 detail 첫 항목에서 loc를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>invalid = client.post(&quot;/ask&quot;, json={&quot;question&quot;: &quot;과제는 어디에 내나요?&quot;, &quot;k&quot;: &quot;2&quot;})
problem = invalid.json()[&quot;detail&quot;][0]
print(invalid.status_code, problem[&quot;loc&quot;][-1])</code></pre>
<p>실행 결과</p><pre><code>422 k</code></pre>
<p>JSON에서 2와 "2"는 서로 다른 값입니다. 이 요청은 StrictInt 검사에서 거부되며 정상 응답의 result 필드가 없습니다. k를 정수 2로 보내면 셀 02에서 본 답변 경로에 도달합니다.</p>
</details>
